In [17]:
import pandas as pd
import numpy as np
import pickle

from pathlib import Path

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense
)

In [18]:
DATA_DIR = Path("../data/processed")

dataset = pd.read_csv(DATA_DIR / "chat_dataset.csv")

questions = dataset["question"].tolist()
answers = dataset["answer"].tolist()

print(len(dataset))

138230


In [19]:
tokenizer = Tokenizer(
    filters="",
    lower=False,
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(questions + answers)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary:", vocab_size)

Vocabulary: 77161


In [20]:
question_sequences = tokenizer.texts_to_sequences(questions)
answer_sequences = tokenizer.texts_to_sequences(answers)

In [21]:
MAX_LENGTH = 20

encoder_input = pad_sequences(
    question_sequences,
    maxlen=MAX_LENGTH,
    padding="post"
)

decoder_input = pad_sequences(
    answer_sequences,
    maxlen=MAX_LENGTH,
    padding="post"
)

In [22]:
decoder_output = np.zeros_like(decoder_input)

decoder_output[:, :-1] = decoder_input[:, 1:]

In [23]:
print("Encoder:", encoder_input.shape)
print("Decoder Input:", decoder_input.shape)
print("Decoder Output:", decoder_output.shape)

Encoder: (138230, 20)
Decoder Input: (138230, 20)
Decoder Output: (138230, 20)


In [24]:
EMBEDDING_DIM = 128
LSTM_UNITS = 256

In [25]:
encoder_inputs = Input(shape=(MAX_LENGTH,), name="encoder_inputs")

encoder_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [26]:
decoder_inputs = Input(shape=(MAX_LENGTH,), name="decoder_inputs")

decoder_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding"
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [27]:
decoder_dense = Dense(
    vocab_size,
    activation="softmax",
    name="decoder_output"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [28]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

In [33]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [34]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 20, 128)   │  9,876,608 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 20)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 20, 128)   │  9,876,608 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 20, 256), │    394,240 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 20, 77161) │ 19,830,377 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,372,073 (154.01 MB)

 Trainable params: 40,372,073 (154.01 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
import pickle

with open("../saved_models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [32]:
print(vocab_size)

77161
